In [1]:
# !pip install torch torchvision torchaudio

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [4]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [6]:
print(f"Using device: {device}")

Using device: cpu


In [7]:
model = NeuralNetwork().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [8]:
# Training the model for 5 epochs
epochs = 5
print("Starting training...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for X, y in train_loader:
        # X = images, y = labels
        X, y = X.to(device), y.to(device)

        # Forward pass
        outputs = model(X)
        loss = criterion(outputs, y)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}, Accuracy: {100 - ((running_loss / len(train_loader) )*100):.2f}%")


Starting training...
Epoch [1/5], Loss: 0.3489, Accuracy: 65.11%
Epoch [2/5], Loss: 0.1590, Accuracy: 84.10%
Epoch [3/5], Loss: 0.1094, Accuracy: 89.06%
Epoch [4/5], Loss: 0.0830, Accuracy: 91.70%
Epoch [5/5], Loss: 0.0649, Accuracy: 93.51%


In [9]:
accuracy =100 - ((running_loss / len(train_loader) )*100)
print(f"Training Accuracy: {accuracy:.2f}%")

Training Accuracy: 93.51%


In [10]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for X, y in test_loader:
        X, y = X.to(device), y.to(device)
        outputs = model(X)
        _, predicted = torch.max(outputs.data, 1)
        total += y.size(0)
        correct += (predicted == y).sum().item()

print(f'Test Accuracy: {100 * correct / total:.2f}%')   

Test Accuracy: 97.58%
